<a href="https://colab.research.google.com/github/itgirlhightech/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [1]:
%pip -q install duckdb huggingface_hub


In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [4]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [5]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_62f4a7e64f5e0096,content_17d994b99d470434,252.0,827.0,1.0,41.917435
1,client_62f4a7e64f5e0096,content_9e6d399bb7df2d21,411.0,874.0,4.0,40.678875
2,client_62f4a7e64f5e0096,content_889961fe0fd51a4b,2140.0,2664.0,6.0,7.070556
3,client_62f4a7e64f5e0096,content_7e67ece7486a4851,101.0,183.0,0.0,35.997698
4,client_62f4a7e64f5e0096,content_762f7da095bfd414,141.0,261.0,0.0,12.166371


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [6]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_62f4a7e64f5e0096,content_17d994b99d470434,252.0,827.0,1.0,41.917435,8.0,0.136531,0.239391,1223.0,1353.0,0.903917
1,client_62f4a7e64f5e0096,content_9e6d399bb7df2d21,411.0,874.0,4.0,40.678875,38.0,0.146092,0.336034,900.0,2173.0,0.414174
2,client_62f4a7e64f5e0096,content_889961fe0fd51a4b,2140.0,2664.0,6.0,7.070556,42.0,0.064587,0.803004,103.0,1146.0,0.089878
3,client_62f4a7e64f5e0096,content_7e67ece7486a4851,101.0,183.0,0.0,35.997698,4.0,0.384248,0.408115,38.0,87.0,0.436782
4,client_62f4a7e64f5e0096,content_762f7da095bfd414,141.0,261.0,0.0,12.166371,3.0,0.104607,0.860845,15.0,36.0,0.416667


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [7]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.542     0.342     0.420      9389
           1      0.685     0.832     0.752     16162

    accuracy                          0.652     25551
   macro avg      0.614     0.587     0.586     25551
weighted avg      0.633     0.652     0.630     25551



### Model choice

For this week, I chose a Decision Tree classifier as the main modeling approach.

The goal is not to maximize model complexity, but to evaluate whether an interpretable model can provide useful predictive information about content decline beyond the majority-class baseline.

In [8]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (
    data['imp_last30'] < 0.8 * data['imp_prev30']
).astype(int)

feature_cols = [
    'imp_prev30',
    'visible_queries',
    'rare_share',
    'anon_share',
    'top_query_share'
]

model_data = data.dropna(subset=feature_cols)

X = model_data[feature_cols]
y = model_data['is_declining']

### Split Design

The dataset is split into 75% training data and 25% test data using train_test_split with random_state=42. Stratification is applied to preserve the class distribution of the is_declining target across both subsets.

The test set is kept separate from model training and is used for the final comparison between the Decision Tree and the majority-class baseline.

In [9]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = DecisionTreeClassifier(
    random_state=42
).fit(X_tr, y_tr)

In [10]:
print(
    f'base rate (always predict majority): '
    f'{max(y_te.mean(), 1 - y_te.mean()):.3f}'
)

print(
    classification_report(
        y_te,
        model.predict(X_te),
        digits=3
    )
)

base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.425     0.438     0.432      9389
           1      0.668     0.656     0.662     16162

    accuracy                          0.576     25551
   macro avg      0.546     0.547     0.547     25551
weighted avg      0.579     0.576     0.577     25551



### Honest Split by Client

To make the validation more realistic, the Week-5 model is re-evaluated using a client-level grouped split.

Instead of randomly splitting individual observations, entire clients are assigned to either the training or test set. This prevents observations from the same client from appearing in both subsets and reduces the risk of overly optimistic evaluation caused by client-level overlap.

The Decision Tree, target variable, and feature set remain unchanged from Week 5. Only the validation design is changed.

In [11]:
from sklearn.model_selection import GroupShuffleSplit

groups = model_data['client_hash_id']

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_tr_grouped = X.iloc[train_idx]
X_te_grouped = X.iloc[test_idx]

y_tr_grouped = y.iloc[train_idx]
y_te_grouped = y.iloc[test_idx]

In [13]:
grouped_model = DecisionTreeClassifier(
    random_state=42
).fit(X_tr_grouped, y_tr_grouped)

print(
    f'base rate (always predict majority): '
    f'{max(y_te_grouped.mean(), 1 - y_te_grouped.mean()):.3f}'
)

print(
    classification_report(
        y_te_grouped,
        grouped_model.predict(X_te_grouped),
        digits=3
    )
)

base rate (always predict majority): 0.677
              precision    recall  f1-score   support

           0      0.359     0.486     0.413     16706
           1      0.705     0.585     0.639     34996

    accuracy                          0.553     51702
   macro avg      0.532     0.536     0.526     51702
weighted avg      0.593     0.553     0.566     51702



The Decision Tree achieved lower accuracy under the client-level grouped split (0.553) than under the Week-5 random split (0.576). The grouped evaluation also produced a higher majority-class baseline (0.677 versus 0.633), indicating a different class distribution in the test set. This suggests that the Week-5 random split may have provided a more favorable evaluation setting, and that model performance should be interpreted cautiously when generalizing across clients.

### Before vs. After Validation

The Week-5 random split and the client-level grouped split use the same target, features, and Decision Tree model. The main difference is the validation design.

The grouped split results in lower accuracy and F1 than the original random split, while also producing a higher majority-class baseline. This indicates that the validation design affects the observed model performance and that the Week-5 result should not be treated as a definitive measure of generalization across clients.

In [14]:
import pandas as pd

comparison = pd.DataFrame({
    'Split': ['Random split (Week 5)', 'Grouped split (Week 6)'],
    'Baseline': [0.633, 0.677],
    'Accuracy': [0.576, 0.553],
    'F1 (class 1)': [0.662, 0.639]
})

comparison

,Split,Baseline,Accuracy,F1 (class 1)
0,Random split (Week 5),0.633,0.576,0.662
1,Grouped split (Week 6),0.677,0.553,0.639


## 3. Leakage Audit

The target `is_declining` is defined using impressions from the last 30 days compared with the previous 30 days.

The feature set was reviewed to check whether any feature directly uses information from the target period.

- `imp_prev30`: historical impressions from the previous 30-day period.
- `visible_queries`: query visibility signal.
- `rare_share`: share of impressions from rare queries.
- `anon_share`: share of anonymized impressions.
- `top_query_share`: concentration of impressions in the top query.

The model does not directly use `imp_last30` as a feature. Therefore, the target outcome is not explicitly included among the predictors.

However, the query-based features require careful interpretation because their aggregation window should be confirmed to occur before the prediction period. This audit therefore treats the absence of direct target leakage as supported, while recognizing that feature timing is an important remaining validation point.

### Leakage Findings

The target `is_declining` is calculated from `imp_last30`, representing the most recent 30-day period in the dataset.

The `imp_prev30` feature is calculated from the preceding period and therefore does not directly overlap with the target window.

The remaining query-based features (`visible_queries`, `rare_share`, `anon_share`, and `top_query_share`) are sourced from `fact_query_90d`. Their exact temporal construction could not be confirmed from the available notebook cells, so their absence of temporal leakage cannot be fully established from this audit alone.

Therefore, no direct leakage was identified in the target/`imp_prev30` construction, but the timing of the query-based features remains an uncertainty.

In [15]:
y_pred_grouped = grouped_model.predict(X_te_grouped)

failure_examples = model_data.iloc[test_idx].copy()
failure_examples['actual'] = y_te_grouped.values
failure_examples['predicted'] = y_pred_grouped

failures = failure_examples[
    failure_examples['actual'] != failure_examples['predicted']
]

print(f"Total failures: {len(failures):,}")

failures.head(10)


Total failures: 23,106


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share,is_declining,actual,predicted
1,client_62f4a7e64f5e0096,content_9e6d399bb7df2d21,411.0,874.0,4.0,40.678875,38.0,0.146092,0.336034,900.0,2173.0,0.414174,1,1,0
4,client_62f4a7e64f5e0096,content_762f7da095bfd414,141.0,261.0,0.0,12.166371,3.0,0.104607,0.860845,15.0,36.0,0.416667,1,1,0
5,client_62f4a7e64f5e0096,content_9551aad28ed05868,1745.0,2336.0,6.0,10.904079,76.0,0.070223,0.637634,431.0,3324.0,0.129663,1,1,0
7,client_62f4a7e64f5e0096,content_cd4e5e7a4c5d80fb,202.0,584.0,1.0,4.615049,15.0,0.135701,0.721121,62.0,383.0,0.161880,1,1,0
8,client_62f4a7e64f5e0096,content_8c475ac85f10a360,160.0,156.0,0.0,80.767842,3.0,0.735795,0.122159,19.0,50.0,0.380000,0,0,1
9,client_62f4a7e64f5e0096,content_c08d7c3812447f8c,422.0,170.0,0.0,72.152033,12.0,0.555723,0.087349,52.0,237.0,0.219409,0,0,1
12,client_62f4a7e64f5e0096,content_da5184015fb80f89,539.0,1393.0,1.0,8.729105,32.0,0.076167,0.805833,79.0,708.0,0.111582,1,1,0
14,client_62f4a7e64f5e0096,content_78a88aecd1f6118a,298.0,369.0,0.0,13.850601,8.0,0.149227,0.737812,51.0,190.0,0.268421,0,0,1
15,client_62f4a7e64f5e0096,content_e63e7328c44524b8,30696.0,29265.0,118.0,5.337824,230.0,0.019888,0.804401,800.0,17096.0,0.046795,0,0,1
16,client_62f4a7e64f5e0096,content_72447e3bef258081,15072.0,8081.0,55.0,8.712147,124.0,0.042159,0.643373,2682.0,10107.0,0.265361,0,0,1


In [16]:
failures[['client_hash_id', 'content_hash_id',
          'imp_prev30', 'visible_queries',
          'rare_share', 'anon_share',
          'top_query_share',
          'actual', 'predicted']].head(10)

,client_hash_id,content_hash_id,imp_prev30,visible_queries,rare_share,anon_share,top_query_share,actual,predicted
1,client_62f4a7e64f5e0096,content_9e6d399bb7df2d21,874.0,38.0,0.146092,0.336034,0.414174,1,0
4,client_62f4a7e64f5e0096,content_762f7da095bfd414,261.0,3.0,0.104607,0.860845,0.416667,1,0
5,client_62f4a7e64f5e0096,content_9551aad28ed05868,2336.0,76.0,0.070223,0.637634,0.129663,1,0
7,client_62f4a7e64f5e0096,content_cd4e5e7a4c5d80fb,584.0,15.0,0.135701,0.721121,0.161880,1,0
8,client_62f4a7e64f5e0096,content_8c475ac85f10a360,156.0,3.0,0.735795,0.122159,0.380000,0,1
9,client_62f4a7e64f5e0096,content_c08d7c3812447f8c,170.0,12.0,0.555723,0.087349,0.219409,0,1
12,client_62f4a7e64f5e0096,content_da5184015fb80f89,1393.0,32.0,0.076167,0.805833,0.111582,1,0
14,client_62f4a7e64f5e0096,content_78a88aecd1f6118a,369.0,8.0,0.149227,0.737812,0.268421,0,1
15,client_62f4a7e64f5e0096,content_e63e7328c44524b8,29265.0,230.0,0.019888,0.804401,0.046795,0,1
16,client_62f4a7e64f5e0096,content_72447e3bef258081,8081.0,124.0,0.042159,0.643373,0.265361,0,1


## 4. Real Failure Examples

The grouped evaluation produced both false negatives and false positives.

False negatives show cases where the content was actually classified as declining, but the model predicted the opposite. False positives show cases where the model predicted decline even though the observed outcome did not meet the decline definition.

For example, some false negatives had relatively high previous-period impressions and different query-signal profiles, while false positives also occurred across a range of feature values.

These examples show that the available features do not provide a sufficiently consistent separation between the two target classes. The failure cases should therefore be interpreted as evidence of model limitations rather than as proof that any individual feature caused the prediction error.


## 5. Claim Rewrite

### Original claim

The Decision Tree identified relevant predictive patterns among the available features.

### Revised claim

The Decision Tree showed directional associations between the available features and the observed decline label in this evaluation. However, performance decreased under the client-level grouped split, and the model produced both false positives and false negatives. Therefore, the results should be treated as directional and decision-support evidence rather than as proof of reliable prediction or generalization across clients.

## 6. Self-check

- [x] The model was evaluated using a client-level grouped split.
- [x] The same Decision Tree, target, and feature set were maintained for the before/after comparison.
- [x] The grouped split was compared with the original Week-5 random split.
- [x] Both false positives and false negatives were inspected.
- [x] Direct target leakage through `imp_last30` was not identified in the feature construction.
- [x] The temporal construction of the query-based features could not be fully verified from the available notebook code.
- [x] Claims were rewritten using cautious language such as observed, directional, and decision-support.
- [x] The results are not presented as proof of reliable generalization or causality.

## 1. Two Paper Findings + My Methodology Questions

### Finding 1 — Content Refresh

The paper reports that 365+ day content refreshed within the last 30 days showed a 3.2x health-score increase and 57x more impressions compared with the referenced comparison group.

**Methodology question:** Could this observed difference be influenced by selection effects or other characteristics of the pages that were refreshed, rather than the refresh itself? The result is observational, so the measured association should not automatically be interpreted as a causal effect.

### Finding 2 — AI Traffic

The paper reports that high-AI-traffic pages had approximately 9x more impressions while showing a weaker average Google position than pages without AI traffic. The paper interprets this as evidence that AI-referral visibility is behaviorally different from classic organic search visibility.

**Methodology question:** How complete is the measurement of AI visibility, given that known-referral rules may miss some AI sources? Could incomplete referral detection affect the observed relationship between AI traffic and search performance?